# 02 · BindingDB analysis (Ki) — 5-HT7 receptor, acetylcholinesterase, Bcl-2

In [8]:
# Imports
import os
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

In [9]:

# Choose your target here = name of the TSV file downloaded from BindingDB, without ".tsv"
TARGET = "Bcl-2"           # REVISION: one name for all files ("5-HT7_receptor", "Acetylcholinesterase", "Bcl-2")

# Global parameters
ENDPOINTS = ["Ki"]                  # Use only Ki measurements
THR_CLASS = 100                     # Classification threshold (nM)
CONVERT_TO_LOG = False              # values stay in nM (REVISION: the duplicated, contradictory first block was removed)
RM_HIGH_STD = True                  # Remove compounds whose replicate SD >= 10 nM (REVISION: now actually applied in Part 3)
ORGANISM = "Homo sapiens"           # REVISION: keep human target records only

# File paths
RAW_INPUT_TSV = f"{TARGET}.csv"     # the BindingDB download. If you only have your earlier extract, write its
                                   # name here instead, e.g. "Acetylcholinesterase.csv" or "Apoptosis_regulator_Bcl-2.csv"
RAW_CSV = f"{TARGET}_raw.csv"      # REVISION: new name, so the input file is never overwritten
STRUCT_FILTERED_CSV = f"{TARGET}_filter.csv"
CURATED_CSV = f"{TARGET}_curated.csv"
FINAL_CSV = f"{TARGET}_final.csv"


In [10]:

# Load raw BindingDB file
# read either the BindingDB download (.tsv, tab-separated) or an earlier extract (.csv, comma-separated)
sep = "\t" if RAW_INPUT_TSV.lower().endswith((".tsv")) else ","
df = pd.read_csv(RAW_INPUT_TSV, sep=sep, dtype=str, low_memory=False)
print("File:", RAW_INPUT_TSV, "| records:", len(df), "| columns:", list(df.columns)[:8], "...")

# REVISION (editor comment 4): keep records of the human target only
ORGANISM_COL = "Target Source Organism According to Curator or DataSource"
if ORGANISM_COL in df.columns:
    human = df[ORGANISM_COL].astype(str).str.strip() == ORGANISM
    print("Removed non-human records:", int((~human).sum()), df.loc[~human, ORGANISM_COL].value_counts().to_dict())
    df = df[human]
else:
    print("WARNING: no organism column in", RAW_INPUT_TSV, "- organism could not be checked")

# Column mapping
# REVISION: two layouts are recognised - the BindingDB download, and your earlier extracts
COLUMN_MAPS = [
    {"BindingDB Reactant_set_id": "BindingDB_ID", "Ligand SMILES": "smiles", "Ki (nM)": "value"},   # BindingDB .tsv download
    {"BindingDB_ID": "BindingDB_ID", "smiles": "smiles", "value": "value"},                          # 5-HT7 / AChE extract
    {"bindingdb_id": "BindingDB_ID", "smiles": "smiles", "value": "value"},                          # Bcl-2 extract
]
COLUMN_MAP = next((m for m in COLUMN_MAPS if all(col in df.columns for col in m)), COLUMN_MAPS[0])

# stop with a clear message if a column name in your download is different
missing_cols = [col for col in COLUMN_MAP if col not in df.columns]
if missing_cols:
    raise KeyError(f"Column(s) {missing_cols} not found in {RAW_INPUT_TSV}.\n"
                   f"Columns in your file: {list(df.columns)}\n"
                   "Edit the left-hand names in COLUMN_MAPS so they match your file exactly.")

df = df[list(COLUMN_MAP.keys())].rename(columns=COLUMN_MAP)

# Add standard type
df["standard_type"] = "Ki (nM)"

# censored values (e.g. ">10000", "<0.001") are removed explicitly.
# Before, pd.to_numeric(errors="coerce") turned them into empty values without any visible step.
text = df["value"].astype(str).str.strip()
censored_lt, censored_gt = text.str.startswith("<"), text.str.startswith(">")
print("Removed censored Ki values: '<'", int(censored_lt.sum()), "| '>'", int(censored_gt.sum()))
df = df[~(censored_lt | censored_gt)].copy()

df["value"] = pd.to_numeric(df["value"].astype(str).str.strip(), errors="coerce")
print("Records without a Ki value (removed in Part 3):", int(df["value"].isna().sum()))

print("Raw dataset shape:", df.shape)

df.to_csv(RAW_CSV, index=False)
print(df.head())


File: Bcl-2.csv | records: 3784 | columns: ['smiles', 'standard_type', 'value', 'units', 'bindingdb_id', 'data_validity_comment', 'activity_comment', 'warning_flag'] ...
Removed censored Ki values: '<' 914 | '>' 185
Records without a Ki value (removed in Part 3): 0
Raw dataset shape: (2685, 4)
   BindingDB_ID                                             smiles  value  \
11       324837  CN(C)CCOc1ccc(c(CN2CCN(CC2)c2ccc(C(=O)NS(=O)(=...  0.001   
27       325111  CC1(C)CCC(CN2CCN(CC2)c2ccc(C(=O)NS(=O)(=O)c3cc...  0.002   
28      1372860  CC1(C)CCC(CN2CCN(CC2)c2ccc(C(=O)NS(=O)(=O)c3cc...  0.002   
29       325118  CC1(C)CCC(CN2CCN(CC2)c2ccc(C(=O)NS(=O)(=O)c3cc...  0.002   
30       704721  CC1(C)CCC(CN2CCN(CC2)c2ccc(C(=O)NS(=O)(=O)c3cc...  0.002   

   standard_type  
11       Ki (nM)  
27       Ki (nM)  
28       Ki (nM)  
29       Ki (nM)  
30       Ki (nM)  


In [11]:
# Part 2: Structure preparation (sanitize/neutralize)
def _InitialiseNeutralisationReactions():
    patts = (
        # Imidazoles
        ('[n+;H]', 'n'),
        # Amines
        ('[N+;!H0]', 'N'),
        # Carboxylic acids and alcohols
        ('[$([O-]);!$([O-][#7])]', 'O'),
        # Thiols
        ('[S-;X1]', 'S'),
        # Sulfonamides
        ('[$([N-;X2]S(=O)=O)]', 'N'),
        # Enamines
        ('[$([N-;X2][C,N]=C)]', 'N'),
        # Tetrazoles
        ('[n-]', '[nH]'),
        # Sulfoxides
        ('[$([S-]=O)]', 'S'),
        # Amides
        ('[$([N-]C=O)]', 'N'),
    )
    return [(Chem.MolFromSmarts(x), Chem.MolFromSmiles(y, False)) for x, y in patts]

def sanitize_mol(smiles: str):
    if smiles is None or not isinstance(smiles, str):
        return smiles, True
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles, True
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return smiles, True
    return smiles, False

def neutralize_mol(smiles: str):
    neutralized = False
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles, False
    for reactant, product in _InitialiseNeutralisationReactions():
        while mol.HasSubstructMatch(reactant):
            neutralized = True
            rms = AllChem.ReplaceSubstructs(mol, reactant, product)
            mol = rms[0]
    smiles_out = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    return smiles_out, neutralized

def prepare_structures(smiles: str, remove_salts=True, do_sanitize=True, do_neutralize=True):
    salt = False
    failed = False
    neutralized = False

    if smiles is None or not isinstance(smiles, str) or not smiles.strip():
        return 'missing', False, True, False

    if remove_salts:
        # test only the SMILES part. BindingDB CXSMILES end with "|(8.08,-9.44,;...)|" atom
        # coordinates whose decimal points made real molecules look like salts/mixtures.
        salt = ("." in smiles.split(" ")[0])

    if do_sanitize:
        smiles, failed = sanitize_mol(smiles)

    if do_neutralize and not failed:
        smiles, neutralized = neutralize_mol(smiles)

    return smiles, salt, failed, neutralized

# Load raw data
df_raw = pd.read_csv(RAW_CSV)

# Apply structure preparation
print("Applying structure preparation...")
prep = df_raw['smiles'].apply(lambda s: prepare_structures(s, True, True, True))
df_raw[['CuratedSmiles', 'IsSalt', 'FailedSanit', 'Neutralized']] = pd.DataFrame(
    prep.tolist(), index=df_raw.index
)

# Analysis: what was detected at this step
print("Structure prep analysis:")
total = len(df_raw)
print(" - Input rows:", total)
print(" - IsSalt True:", int(df_raw['IsSalt'].sum()))
print(" - FailedSanit True:", int(df_raw['FailedSanit'].sum()))
print(" - Neutralized True:", int(df_raw['Neutralized'].sum()))
print(" - Unique CuratedSmiles:", df_raw['CuratedSmiles'].nunique())

# Filter: remove salts and failed sanitization ONLY
f1 = df_raw.loc[~df_raw['IsSalt']].copy()
f2 = f1.loc[~f1['FailedSanit']].copy()
struct_df = f2.copy()

# Duplicates overview AFTER filtering
dup_counts = struct_df['CuratedSmiles'].value_counts()
dup_groups = (dup_counts > 1).sum()
dup_extra_rows = int(dup_counts[dup_counts > 1].sum() - dup_groups)

print("\nPost-filter analysis:")
print(" - Remaining rows:", len(struct_df))
print(" - Removed salts:", total - len(f1))
print(" - Removed failed sanitization:", len(f1) - len(f2))
print(" - Duplicate groups:", int(dup_groups))
print(" - Extra duplicate rows:", int(dup_extra_rows))

# Save structure-filtered data
struct_df.to_csv(STRUCT_FILTERED_CSV, index=False)
print(f"\n✓ Saved structure-filtered to: {STRUCT_FILTERED_CSV}\n")

Applying structure preparation...
Structure prep analysis:
 - Input rows: 2685
 - IsSalt True: 0
 - FailedSanit True: 2
 - Neutralized True: 1
 - Unique CuratedSmiles: 1651

Post-filter analysis:
 - Remaining rows: 2683
 - Removed salts: 0
 - Removed failed sanitization: 2
 - Duplicate groups: 636
 - Extra duplicate rows: 1034

✓ Saved structure-filtered to: Bcl-2_filter.csv



In [12]:
# Part 3: Experimental curation (Dixon's Q test, statistics)
def _InitLookUp(alpha=0.05):
    if alpha == 0.10:
        q_tab = [0.941, 0.765, 0.642, 0.56, 0.507, 0.468, 0.437, 0.412, 0.392, 0.376, 0.361, 0.349, 0.338, 0.329,
                 0.32, 0.313, 0.306, 0.3, 0.295, 0.29, 0.285, 0.281, 0.277, 0.273, 0.269, 0.266, 0.263, 0.26]
    elif alpha == 0.05:
        q_tab = [0.97, 0.829, 0.71, 0.625, 0.568, 0.526, 0.493, 0.466, 0.444, 0.426, 0.41, 0.396, 0.384, 0.374,
                 0.365, 0.356, 0.349, 0.342, 0.337, 0.331, 0.326, 0.321, 0.317, 0.312, 0.308, 0.305, 0.301, 0.29]
    elif alpha == 0.01:
        q_tab = [0.994, 0.926, 0.821, 0.74, 0.68, 0.634, 0.598, 0.568, 0.542, 0.522, 0.503, 0.488, 0.475, 0.463,
                 0.452, 0.442, 0.433, 0.425, 0.418, 0.411, 0.404, 0.399, 0.393, 0.388, 0.384, 0.38, 0.376, 0.372]
    else:
        q_tab = []
    return {n: q for n, q in zip(range(3, len(q_tab)+3), q_tab)}

def dixon_test(data, left=True, right=True, alpha=0.05):
    """Perform Dixon's Q test for outlier detection."""
    q_dict = _InitLookUp(alpha=alpha)
    if not (3 <= len(data) <= max(q_dict.keys(), default=0)):
        return []
    
    sdata = sorted(data)
    Q_mindiff, Q_maxdiff = (0, 0), (0, 0)
    
    if left:
        Q_min = (sdata[1] - sdata[0])
        try:
            Q_min /= (sdata[-1] - sdata[0])
            Q_mindiff = (Q_min - q_dict[len(data)], sdata[0])
        except (ZeroDivisionError, KeyError):
            pass
    
    if right:
        Q_max = abs((sdata[-2] - sdata[-1]))
        try:
            Q_max /= abs((sdata[0] - sdata[-1]))
            Q_maxdiff = (Q_max - q_dict[len(data)], sdata[-1])
        except (ZeroDivisionError, KeyError):
            pass

    if not Q_mindiff[0] > 0 and not Q_maxdiff[0] > 0:
        return []
    if Q_mindiff[0] == Q_maxdiff[0]:
        return [Q_mindiff[1], Q_maxdiff[1]]
    if Q_mindiff[0] > Q_maxdiff[0]:
        return [Q_mindiff[1]]
    return [Q_maxdiff[1]]

def compute_stats(values: np.ndarray):
    """Compute mean, std, and SEM for values."""
    n = len(values)
    mean = float(np.mean(values)) if n else np.nan
    std = float(np.std(values, ddof=0)) if n else np.nan
    sem = float(std / np.sqrt(n)) if n else np.nan
    return mean, std, sem, n

def concatenate_string_info(data: pd.DataFrame, outlier_indices: list):
    """Concatenate ChEMBL IDs and other info, excluding outliers."""
    st = data["standard_type"].copy()
    bid = data["BindingDB_ID"].copy()
    mid = data["smiles"].copy()
    
    # Remove outliers by index
    if outlier_indices:
        st = st.drop(st.index[outlier_indices], errors='ignore')
        bid = bid.drop(bid.index[outlier_indices], errors='ignore')
        mid = mid.drop(mid.index[outlier_indices], errors='ignore')
    
    standard_types = ",".join(sorted(set(st.astype(str))))
    
    # Convert IDs safely, handling NaN values and scientific notation
    def safe_id_convert(x):
        try:
            if pd.isna(x):
                return ""
            # Convert float to int first (to remove decimal), then to string
            if isinstance(x, float):
                return str(int(x))
            return str(x)
        except (ValueError, OverflowError):
            return str(x)
    
    bindingdb_ids = ";".join(sorted(set(filter(None, [safe_id_convert(x) for x in bid]))))
    smiles_str = ";".join(sorted(set(mid.astype(str))))
    
    return smiles_str, bindingdb_ids, standard_types


# replicate aggregation now (1) groups replicates on the standardised SMILES (CuratedSmiles),
# (2) removes every Dixon outlier value, (3) applies the replicate-SD filter (RM_HIGH_STD) and
# (4) writes the columns that Part 4 reads (exp_mean, bindingdb_id). This is the same procedure as
# the ChEMBL notebook and reproduces your deposited 5-HT7 and Bcl-2 curated files exactly.
def curate_values(values, outlier_test=True, alpha=0.05):
    new_values = values.copy()
    num_outliers = 0
    outlier_indices = []
    if outlier_test:
        with np.errstate(invalid="ignore", divide="ignore"):   # identical replicates give 0/0 = no outlier
            outliers = dixon_test(list(values), left=True, right=True, alpha=alpha)
        if outliers:
            mask = np.isin(values, np.array(outliers))
            outlier_indices = list(np.where(mask)[0])
            new_values = values[~mask]
            num_outliers = int(mask.sum())
    mean, std, sem, num_values = compute_stats(new_values)
    return mean, std, sem, num_outliers, num_values, new_values, outlier_indices


struct_df = pd.read_csv(STRUCT_FILTERED_CSV)
work = struct_df.copy()
work["value"] = pd.to_numeric(work["value"], errors="coerce")
work = work.dropna(subset=["CuratedSmiles", "value"])

rows, total_outliers, groups_with_outliers = [], 0, 0
for smi, group in work.groupby("CuratedSmiles", sort=False):
    values = group["value"].to_numpy(dtype=float)
    mean, std, sem, n_outliers, n_values, new_values, outlier_indices = curate_values(values, alpha=0.05)
    if n_outliers:
        groups_with_outliers += 1
        total_outliers += n_outliers
    _, bindingdb_ids, standard_types = concatenate_string_info(group, outlier_indices)
    rows.append({
        "smiles": smi,
        "exp_mean": mean,
        "exp_std": std,
        "exp_sem": sem,
        "no.entries": int(len(values)),
        "no.outliers": int(n_outliers),
        "standard_types": standard_types,
        "bindingdb_id": bindingdb_ids,
    })

curated_df = pd.DataFrame(rows)
before = len(curated_df)
if RM_HIGH_STD:
    curated_df = curated_df.loc[~(curated_df["exp_std"] >= 10)].copy()

print("Experimental curation analysis:")
print(" - Records without a Ki value removed:", int(struct_df["value"].isna().sum()))
print(" - Unique compounds (CuratedSmiles):", before)
print(" - Compounds with outliers:", groups_with_outliers, "| outlier values removed:", total_outliers)
print(" - Removed high-std compounds (std >= 10 nM):", before - len(curated_df))
print(" - Final compounds:", len(curated_df))

curated_df.to_csv(CURATED_CSV, index=False)
print("Saved curated to:", CURATED_CSV)


Experimental curation analysis:
 - Records without a Ki value removed: 0
 - Unique compounds (CuratedSmiles): 1649
 - Compounds with outliers: 34 | outlier values removed: 34
 - Removed high-std compounds (std >= 10 nM): 13
 - Final compounds: 1636
Saved curated to: Bcl-2_curated.csv


In [13]:
def class_vector(table_final: pd.DataFrame, thr_class: float):
    """Assign activity class based on threshold."""
    out = pd.DataFrame({
        'smiles': table_final['smiles'],
        'bindingdb_id': table_final['bindingdb_id'],
        'exp_mean [nM]': table_final['exp_mean'],
    })
    out['class'] = table_final['exp_mean'] < thr_class
    return out

# Load curated data
exp_df = pd.read_csv(CURATED_CSV)

final_df = class_vector(exp_df, THR_CLASS)

# Analysis
n = len(final_df)
n_active = int(final_df['class'].sum())
print("Class assignment analysis:")
print(" - Total compounds:", n)
print(" - Active (exp_mean < {} nM):".format(THR_CLASS), n_active)
print(" - Inactive:", n - n_active)
print(" - % Active:", round(100.0 * n_active / n, 2) if n > 0 else 0)

print("\nfinal_df.columns:", list(final_df.columns))
print(final_df.head(10))

# Save final data
final_df.to_csv(FINAL_CSV, index=False)
print(f"\n✓ Saved final to: {FINAL_CSV}\n")

Class assignment analysis:
 - Total compounds: 1636
 - Active (exp_mean < 100 nM): 1110
 - Inactive: 526
 - % Active: 67.85

final_df.columns: ['smiles', 'bindingdb_id', 'exp_mean [nM]', 'class']
                                              smiles  \
0  CN(C)CCOc1ccc(-c2ccc(Cl)cc2)c(CN2CCN(c3ccc(C(=...   
1  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(OC...   
2  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(OC...   
3  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...   
4  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4cnc(OC...   
5  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...   
6  CN(C)CC(=O)N1CCO[C@H](COc2ncc(S(=O)(=O)NC(=O)c...   
7  CN(C)CCOc1ccc(-c2ccc(Cl)cc2)c(CN2CCN(c3ccc(C(=...   
8  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...   
9  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(N[...   

                    bindingdb_id  exp_mean [nM]  class  
0                         324837          0.001   True  
1                         325111          0.002   True  
2  1180089;13728

In [14]:
#import library
import pandas as pd
import numpy as np
from pathlib import Path
import os
from datetime import datetime

# Import MoleculeACE
from MoleculeACE.benchmark.data_prep import split_data
from MoleculeACE.benchmark.utils import Data
from MoleculeACE.benchmark.const import Descriptors
from MoleculeACE.models import RF
from MoleculeACE.benchmark.utils import calc_rmse, calc_cliff_rmse


# Setup directories
PROJECT_ROOT = Path(os.getcwd()).resolve()
DATA_DIR = PROJECT_ROOT / 'Data'
RESULTS_DIR = PROJECT_ROOT / 'Results'
PROCESS_DIR = PROJECT_ROOT / 'Process_Steps'
# Create all directories
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
PROCESS_DIR.mkdir(exist_ok=True)

In [15]:

# Import MoleculeACE
from MoleculeACE.benchmark.data_prep import split_data

# Load your CSV file
input_file = FINAL_CSV                      #  follows TARGET
df_raw = pd.read_csv(input_file)
print(f" Loaded: {input_file}")
print(f"  Shape: {df_raw.shape}")
print(f"  Columns: {df_raw.columns.tolist()}")
print(f"  Sample:\n{df_raw.head(3).to_string()}\n")

# Save step 0
step0_file = PROCESS_DIR / f'{TARGET}_final.csv'
df_raw.to_csv(step0_file, index=False)

# Check bioactivity column (should be 'exp_mean [nM]')
bioactivity_col = 'exp_mean [nM]'

# Step 1: Remove NaN values
df_clean = df_raw.dropna(subset=[bioactivity_col]).copy()
print(f"✓ Removed NaN: {len(df_raw) - len(df_clean)} rows")

# Step 2: Remove zero and negative values
df_clean = df_clean[df_clean[bioactivity_col] > 0].copy()
print(f"✓ Removed zero/negative bioactivity: {len(df_raw) - len(df_clean)} rows")

# Step 3: Check for SMILES column
if 'smiles' not in df_clean.columns:
    print("✗ ERROR: 'smiles' column not found!")
    print(f"Available columns: {df_clean.columns.tolist()}")
    raise ValueError("Missing 'smiles' column")

# Step 4: Remove invalid SMILES
df_clean = df_clean[df_clean['smiles'].notna()].copy()
df_clean = df_clean[df_clean['smiles'].str.strip() != ''].copy()
print(f"✓ Removed invalid SMILES: {len(df_raw) - len(df_clean)} rows total")

print(f"\nAfter cleaning:")
print(f"  Shape: {df_clean.shape}")
print(f"  Bioactivity stats:\n{df_clean[bioactivity_col].describe()}")
print(f"  Min bioactivity: {df_clean[bioactivity_col].min():.2e} nM")
print(f"  Max bioactivity: {df_clean[bioactivity_col].max():.2e} nM\n")

# Save cleaned data
clean_file = PROCESS_DIR / f'{TARGET}_cleaned.csv'
df_clean.to_csv(clean_file, index=False)
print(f"✓ Cleaned data saved: {clean_file}\n")

# Extract SMILES and bioactivity
smiles_list = df_clean['smiles'].tolist()
bioactivity = df_clean[bioactivity_col].tolist()
print(f"✓ Extracted {len(smiles_list)} molecules")
print(f"  Bioactivity range: {min(bioactivity):.2e} - {max(bioactivity):.2e} nM\n")

# Use MoleculeACE split_data
df_prepared = split_data(
    smiles=smiles_list,
    bioactivity=bioactivity,  # ← Just the values, not the column name!
    n_clusters=5,
    test_size=0.2,
    similarity=0.9,
    potency_fold=10,
    remove_stereo=True
)

prepared_file = DATA_DIR / f'{TARGET}_prepared.csv'
df_prepared.to_csv(prepared_file, index=False)

print(f" Data prepared!")
print(f"  Total: {len(df_prepared)}")
print(f"  Train: {(df_prepared['split'] == 'train').sum()}")
print(f"  Test: {(df_prepared['split'] == 'test').sum()}")
print(f"  Cliff molecules: {df_prepared['cliff_mol'].sum()}")

 Loaded: Bcl-2_final.csv
  Shape: (1636, 4)
  Columns: ['smiles', 'bindingdb_id', 'exp_mean [nM]', 'class']
  Sample:
                                                                                                                            smiles                   bindingdb_id  exp_mean [nM]  class
0              CN(C)CCOc1ccc(-c2ccc(Cl)cc2)c(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NCCCN5CCCC5)c([N+](=O)[O-])c4)c(Oc4ccccc4)c3)CC2)c1                         324837          0.001   True
1        CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(OCC5(F)CCOCC5)c([N+](=O)[O-])c4)c(Oc4cccc5[nH]cnc45)c3)CC2)=C(c2ccc(Cl)cc2)C1                         325111          0.002   True
2  CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(OCC5CCN(C(CF)CF)CC5)c([N+](=O)[O-])c4)c(Oc4cnc5[nH]ccc5c4)c3)CC2)=C(c2ccc(Cl)cc2)C1  1180089;1372860;339678;704721          0.002   True

✓ Removed NaN: 0 rows
✓ Removed zero/negative bioactivity: 0 rows
✓ Removed invalid SMILES: 0 rows total

After cleaning:
  Shape: (1636, 4)
  Bi

100%|██████████| 1636/1636 [00:02<00:00, 652.46it/s] 


Removed 154 stereoisomers


100%|██████████| 1482/1482 [00:02<00:00, 722.50it/s] 
[00:25:41] Explicit valence for atom # 34 C, 6, is greater than permitted
[00:25:41] Explicit valence for atom # 35 C, 6, is greater than permitted


Could not create a generic scaffold of CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NCC5CCOCC5)c(S(F)(F)(F)(F)F)c4)c(Oc4cnc5[nH]ccc5c4)c3)CC2)=C(c2ccc(Cl)cc2)C1, used a normal scaffold instead
Could not create a generic scaffold of CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NCC5CCOCC5)c(CS(F)(F)(F)(F)F)c4)c(Oc4cnc5[nH]ccc5c4)c3)CC2)=C(c2ccc(Cl)cc2)C1, used a normal scaffold instead


100%|██████████| 1482/1482 [00:02<00:00, 739.69it/s] 


 Data prepared!
  Total: 1482
  Train: 1184
  Test: 298
  Cliff molecules: 909


**Demonstration model only.** The random forest below checks that the prepared file loads and trains.
It is *not* used for any result in the paper; models are selected and evaluated in
`04_rerun_with_validation.ipynb`.

In [16]:
# STEP 2: Load and Featurize

data = Data(str(prepared_file))

data.featurize_data(Descriptors.ECFP, radius=2, nbits=1024)

print(f" Featurized!")
print(f"  Descriptor: ECFP (Morgan Fingerprints)")
print(f"  Radius: 2")
print(f"  nBits: 1024")
print(f"  Train features: {data.x_train.shape}")
print(f"  Test features: {data.x_test.shape}\n")

#Train Model
model = RF(n_estimators=100, random_state=42)
model.train(data.x_train, data.y_train, None, None)
print(f" Model trained!")
print(f"  Algorithm: Random Forest")
print(f"  n_estimators: 100")
print(f"  Training samples: {len(data.y_train)}\n")

# STEP 3: PREDICTIONS ON BOTH TRAIN & TEST
# Predict on TRAIN data
y_pred_train = model.predict(data.x_train)
print(f"✓ Train predictions: {len(y_pred_train)} samples")

# Predict on TEST data
y_pred_test = model.predict(data.x_test)
print(f"✓ Test predictions: {len(y_pred_test)} samples\n")

# Train metrics
rmse_train = calc_rmse(data.y_train, y_pred_train)
rmse_cliff_train = calc_cliff_rmse(
    y_test_pred=y_pred_train,
    y_test=data.y_train,
    cliff_mols_test=data.cliff_mols_train
)
mae_train = np.mean(np.abs(np.array(data.y_train) - np.array(y_pred_train)))
r2_train = 1 - (np.sum((np.array(data.y_train) - np.array(y_pred_train))**2) / 
                 np.sum((np.array(data.y_train) - np.mean(data.y_train))**2))

# Test metrics
rmse_test = calc_rmse(data.y_test, y_pred_test)
rmse_cliff_test = calc_cliff_rmse(
    y_test_pred=y_pred_test,
    y_test=data.y_test,
    cliff_mols_test=data.cliff_mols_test
)
mae_test = np.mean(np.abs(np.array(data.y_test) - np.array(y_pred_test)))
r2_test = 1 - (np.sum((np.array(data.y_test) - np.array(y_pred_test))**2) / 
                np.sum((np.array(data.y_test) - np.mean(data.y_test))**2))

print(f" TRAIN METRICS:")
print(f"  RMSE: {rmse_train:.6f}")
print(f"  RMSE (Cliffs): {rmse_cliff_train:.6f}")
print(f"  MAE: {mae_train:.6f}")
print(f"  R²: {r2_train:.6f}")

print(f"\n TEST METRICS:")
print(f"  RMSE: {rmse_test:.6f}")
print(f"  RMSE (Cliffs): {rmse_cliff_test:.6f}")
print(f"  MAE: {mae_test:.6f}")
print(f"  R²: {r2_test:.6f}\n")



 Featurized!
  Descriptor: ECFP (Morgan Fingerprints)
  Radius: 2
  nBits: 1024
  Train features: (1184, 1024)
  Test features: (298, 1024)

 Model trained!
  Algorithm: Random Forest
  n_estimators: 100
  Training samples: 1184

✓ Train predictions: 1184 samples
✓ Test predictions: 298 samples

 TRAIN METRICS:
  RMSE: 0.261289
  RMSE (Cliffs): 0.277405
  MAE: 0.186428
  R²: 0.981919

 TEST METRICS:
  RMSE: 0.711617
  RMSE (Cliffs): 0.721348
  MAE: 0.499372
  R²: 0.876752



In [17]:
#Build Results Function
def build_results(df_split, smiles, cliff_mols, y_true, y_pred, split_name):
    """
    Build results dataframe with all columns including pEC50/pKi
    
    Args:
        df_split: DataFrame with split data (train or test)
        smiles: List of SMILES strings
        cliff_mols: List of cliff molecule indicators
        y_true: True y values
        y_pred: Predicted y values
        split_name: 'train' or 'test'
    
    Returns:
        DataFrame with results
    """
    
    # Safety check
    assert len(df_split) == len(smiles) == len(y_pred), \
        f"Length mismatch in {split_name} split: {len(df_split)} vs {len(smiles)} vs {len(y_pred)}"
    
    results = pd.DataFrame({
        "smiles": smiles,
        "exp_mean [nM]": df_split["exp_mean [nM]"].values,
        "y": df_split["y"].values,
        "cliff_mol": cliff_mols,
        "split": split_name,
    })
    
    # Calculate pEC50 / pKi
    print(f"  Calculating y [pEC50/pKi] for {split_name}...")
    results["y [pEC50/pKi]"] = (
        9 - np.log10(results["exp_mean [nM]"].astype(float))
    )
    
    # Predictions & errors
    results["Predicted_y"] = y_pred
    results["Error"] = np.array(y_true) - np.array(y_pred)
    results["Abs_Error"] = np.abs(results["Error"])
    
    return results


# Get train data
df_train = (
    df_prepared[df_prepared["split"] == "train"]
    .reset_index(drop=True)
)

print(f" Building TRAIN results...")
train_results = build_results(
    df_split=df_train,
    smiles=data.smiles_train,
    cliff_mols=data.cliff_mols_train,
    y_true=data.y_train,
    y_pred=y_pred_train,
    split_name="train"
)
print(f"  Train results: {len(train_results)} rows")

# Get test data
df_test = (
    df_prepared[df_prepared["split"] == "test"]
    .reset_index(drop=True)
)

print(f"✓ Building TEST results...")
test_results = build_results(
    df_split=df_test,
    smiles=data.smiles_test,
    cliff_mols=data.cliff_mols_test,
    y_true=data.y_test,
    y_pred=y_pred_test,
    split_name="test"
)
print(f"  Test results: {len(test_results)} rows\n")

# Concatenate train and test
final_results = (
    pd.concat([train_results, test_results], ignore_index=True)
    .sort_values(["split", "Abs_Error"], ascending=[True, False])
    .reset_index(drop=True)
)

# Save complete results
final_results_file = RESULTS_DIR / f'{TARGET}_predictions.csv'
final_results.to_csv(final_results_file, index=False)
print(f"✓ Saved: {final_results_file}")

 Building TRAIN results...
  Calculating y [pEC50/pKi] for train...
  Train results: 1184 rows
✓ Building TEST results...
  Calculating y [pEC50/pKi] for test...
  Test results: 298 rows

✓ Saved: /home/eagle/Downloads/MTAC_Activity_Cliffs/new_edit/Results/Bcl-2_predictions.csv


In [18]:
# Total molecules
n_total = len(final_results)
n_train = len(train_results)
n_test = len(test_results)

# Cliff molecules
cliff_total = final_results['cliff_mol'].sum()
cliff_train = train_results['cliff_mol'].sum()
cliff_test = test_results['cliff_mol'].sum()

# Percentages
percent_cliff_total = (cliff_total / n_total) * 100 if n_total > 0 else 0
percent_cliff_test = (cliff_test / n_test) * 100 if n_test > 0 else 0

# Create statistics dataframe
statistics = pd.DataFrame({
    'Metric': [
        'n (Total molecules)',
        'nTRAIN (Train molecules)',
        'nTEST (Test molecules)',
        'Cliff (Total cliff molecules)',
        'Cliff TRAIN (Train cliff molecules)',
        'Cliff TEST (Test cliff molecules)',
        '%cliff (% of cliffs in full dataset)',
        '%cliffTRAIN (% of cliffs in train set)',
        '%cliffTEST (% of cliffs in test set)'
    ],
    'Value': [
        n_total,
        n_train,
        n_test,
        cliff_total,
        cliff_train,
        cliff_test,
        f"{percent_cliff_total:.2f}",
        f"{(cliff_train / n_train * 100):.2f}" if n_train > 0 else 0,
        f"{percent_cliff_test:.2f}"
    ]
})

print(f"✓ Statistics calculated:")
print(statistics.to_string(index=False))
print()

# Save as CSV
stats_csv = RESULTS_DIR / f'{TARGET}_statistics.csv'
statistics.to_csv(stats_csv, index=False)
print(f"✓ Saved (CSV): {stats_csv}\n")

✓ Statistics calculated:
                                Metric Value
                   n (Total molecules)  1482
              nTRAIN (Train molecules)  1184
                nTEST (Test molecules)   298
         Cliff (Total cliff molecules)   909
   Cliff TRAIN (Train cliff molecules)   727
     Cliff TEST (Test cliff molecules)   182
  %cliff (% of cliffs in full dataset) 61.34
%cliffTRAIN (% of cliffs in train set) 61.40
  %cliffTEST (% of cliffs in test set) 61.07

✓ Saved (CSV): /home/eagle/Downloads/MTAC_Activity_Cliffs/new_edit/Results/Bcl-2_statistics.csv

